In [1]:
import pandas as pd
import re
import os
import glob
import csv
import xml.etree.ElementTree as ET
from typing import List, Dict, Tuple, Optional
from utilities import *

In [2]:
dna_column_types = ['SNP','RS','DNAMutation','OtherMutation','DNAAllele','DNAAcidChange','Chromosome','GenomicRegion']
pro_column_types = ['ProteinMutation','ProteinAllele','AcidChange','ProteinAcidChange']

In [3]:
####load GT
GT = {}

#tmVar3
gt_tv3 = read_pubtator_format('corpus_groudtruth/tmVar3Corpus.txt')[[0,3,4]]
gt_tv3.columns=['PMID','Mutation','Type']
gt_tv3 = gt_tv3[gt_tv3['Type'].isin(dna_column_types+pro_column_types)]
gt_tv3_dna = gt_tv3[gt_tv3['Type'].isin(dna_column_types)]
gt_tv3_pro = gt_tv3[gt_tv3['Type'].isin(pro_column_types)]
GT['tmVar3'] = gt_tv3
GT['tmVar3_dna'] = gt_tv3_dna
GT['tmVar3_pro'] = gt_tv3_pro

#Wei2013
gt_wei = read_pubtator_format('corpus_groudtruth/Wei2013/test.txt')[[0,3,4]]
gt_wei.columns=['PMID','Mutation','Type']
gt_wei = gt_wei[gt_wei['Type'].isin(dna_column_types+pro_column_types)]
gt_wei_dna = gt_wei[gt_wei['Type'].isin(dna_column_types)]
gt_wei_pro = gt_wei[gt_wei['Type'].isin(pro_column_types)]
GT['Wei2013'] = gt_wei
GT['Wei2013_dna'] = gt_wei_dna
GT['Wei2013_pro'] = gt_wei_pro

#SETH
gt_seth = pd.read_csv('corpus_groudtruth/SETH-corpus/annotations_combined.tsv',sep='\t',header=0)
gt_seth=gt_seth.rename(columns={'text':'Mutation','entity_type':'Type'})
gt_seth = gt_seth[gt_seth['Type'].isin(dna_column_types+pro_column_types)]
GT['SETH'] = gt_seth

#OSIRIS
gt_osiris,_ = load_seth_corpus_xml_labels('corpus_groudtruth/OSIRIS/corpus.xml')
#print('Type for OSIRIS:',gt_osiris['Type'].unique())
gt_osiris = gt_osiris[gt_osiris['Type']=='Variant']
gt_osiris=gt_osiris.rename(columns={'v_lex':'Mutation'})
GT['OSIRIS'] = gt_osiris

GT.keys()

dict_keys(['tmVar3', 'tmVar3_dna', 'tmVar3_pro', 'Wei2013', 'Wei2013_dna', 'Wei2013_pro', 'SETH', 'OSIRIS'])

In [4]:
###load PubMind output
PM = {}

#tmVar3
pm_tv3 = pd.read_csv('results/PubMind/tmVar3.pubmind.out.raw.tsv',sep='\t',index_col=0)
pm_tv3_dna = pm_tv3[pm_tv3['DNA mutation']!='-'][['PMID','DNA mutation']]\
    .drop_duplicates().rename(columns={'DNA mutation':'Mutation'})
pm_tv3_pro = pm_tv3[pm_tv3['protein mutation']!='-'][['PMID','protein mutation']]\
    .drop_duplicates().rename(columns={'protein mutation':'Mutation'})
PM['tmVar3_dna']=pm_tv3_dna
PM['tmVar3_pro']=pm_tv3_pro

#Wei2013
pm_wei = pd.read_csv('results/PubMind/Wei2013_test.pubmind.out.raw.tsv',sep='\t',index_col=0)
pm_wei_dna = pm_wei[pm_wei['DNA mutation']!='-'][['PMID','DNA mutation']]\
    .drop_duplicates().rename(columns={'DNA mutation':'Mutation'})
pm_wei_pro = pm_wei[pm_wei['protein mutation']!='-'][['PMID','protein mutation']]\
    .drop_duplicates().rename(columns={'protein mutation':'Mutation'})
PM['Wei2013_dna']=pm_wei_dna
PM['Wei2013_pro']=pm_wei_pro

#SETH
pm_wei = pd.read_csv('results/PubMind/SETH.pubmind.out.raw.tsv',sep='\t',index_col=0)
pm_wei_dna = pm_wei[pm_wei['DNA mutation']!='-'][['PMID','DNA mutation']]\
    .drop_duplicates().rename(columns={'DNA mutation':'Mutation'})
pm_wei_pro = pm_wei[pm_wei['protein mutation']!='-'][['PMID','protein mutation']]\
    .drop_duplicates().rename(columns={'protein mutation':'Mutation'})
pm_seth_merge = pd.concat([pm_wei_dna,pm_wei_pro]).drop_duplicates().reset_index(drop=True)
PM['SETH'] = pm_seth_merge

#SETH
pm_wei = pd.read_csv('results/PubMind/OSIRIS.pubmind.out.raw.tsv',sep='\t',index_col=0)
pm_wei_dna = pm_wei[pm_wei['DNA mutation']!='-'][['PMID','DNA mutation']]\
    .drop_duplicates().rename(columns={'DNA mutation':'Mutation'})
pm_wei_pro = pm_wei[pm_wei['protein mutation']!='-'][['PMID','protein mutation']]\
    .drop_duplicates().rename(columns={'protein mutation':'Mutation'})
pm_osiris_merge = pd.concat([pm_wei_dna,pm_wei_pro]).drop_duplicates().reset_index(drop=True)
PM['OSIRIS'] = pm_osiris_merge
PM.keys()

dict_keys(['tmVar3_dna', 'tmVar3_pro', 'Wei2013_dna', 'Wei2013_pro', 'SETH', 'OSIRIS'])

In [5]:
####load tmVar3 output (only RefSeq was removed from benchmark as it is Gene ID)
TV3 = {}

#tmVar3
tmv3_tv3 = read_pubtator_format('results/tmVar3/tmVar3Corpus_text_only.bioc.xml.PubTator')[[0,3,4]]
tmv3_tv3.columns = ['PMID','Mutation','Type']
#print('Types for tmVar3 output (\'RefSeq\' removed):',tmv3_tv3['Type'].unique())
tmv3_tv3_dna = tmv3_tv3[tmv3_tv3['Type'].isin(dna_column_types)]
tmv3_tv3_pro = tmv3_tv3[tmv3_tv3['Type'].isin(pro_column_types)]
TV3['tmVar3_dna'] = tmv3_tv3_dna
TV3['tmVar3_pro'] = tmv3_tv3_pro

#Wei2013
tmv3_tv3 = read_pubtator_format('results/tmVar3/Wei2013_test_text_only.bioc.xml.PubTator')[[0,3,4]]
tmv3_tv3.columns = ['PMID','Mutation','Type']
tmv3_tv3_dna = tmv3_tv3[tmv3_tv3['Type'].isin(dna_column_types)]
tmv3_tv3_pro = tmv3_tv3[tmv3_tv3['Type'].isin(pro_column_types)]
TV3['Wei2013_dna'] = tmv3_tv3_dna
TV3['Wei2013_pro'] = tmv3_tv3_pro

#SETH
tmv3_tv3 = read_pubtator_format('results/tmVar3/SETH-corpus.bioc.xml.PubTator')[[0,3,4]]
tmv3_tv3.columns = ['PMID','Mutation','Type']
tmv3_tv3 = tmv3_tv3[tmv3_tv3['Type'].isin(dna_column_types+pro_column_types)]
TV3['SETH'] = tmv3_tv3

#OSIRIS
tmv3_tv3 = read_pubtator_format('results/tmVar3/OSIRIS_text_only.bioc.xml.PubTator')[[0,3,4]]
tmv3_tv3.columns = ['PMID','Mutation','Type']
tmv3_tv3 = tmv3_tv3[tmv3_tv3['Type'].isin(dna_column_types+pro_column_types)]
TV3['OSIRIS'] = tmv3_tv3

TV3.keys()

dict_keys(['tmVar3_dna', 'tmVar3_pro', 'Wei2013_dna', 'Wei2013_pro', 'SETH', 'OSIRIS'])

In [6]:
####load SETH output
SETH = {}

#tmVar3
seth_tv3 = pd.read_csv('results/SETH/tmVar3.seth.out.tsv',sep='\t').rename(columns={'text':'Mutation'})[['PMID','Mutation']]
SETH['tmVar3'] = seth_tv3

#Wei2013
seth_tv3 = pd.read_csv('results/SETH/Wei2013_test.seth.out.tsv',sep='\t').rename(columns={'text':'Mutation'})[['PMID','Mutation']]
SETH['Wei2013'] = seth_tv3

#SETH
seth_tv3 = pd.read_csv('results/SETH/SETH.seth.out.tsv',sep='\t').rename(columns={'text':'Mutation'})[['PMID','Mutation']]
SETH['SETH'] = seth_tv3

#OSIRIS
seth_tv3 = pd.read_csv('results/SETH/OSIRIS.seth.out.tsv',sep='\t').rename(columns={'text':'Mutation'})[['PMID','Mutation']]
SETH['OSIRIS'] = seth_tv3

SETH.keys()

dict_keys(['tmVar3', 'Wei2013', 'SETH', 'OSIRIS'])

In [7]:
####load AIONER output
AIONER = {}

#tmVar3
aioner_out = read_pubtator_format('results/AIONER/tmVar3Corpus_text_only.txt')[[0,3,4]]
aioner_out.columns = ['PMID','Mutation','Type']
aioner_out = aioner_out[aioner_out['Type']=='Variant']
AIONER['tmVar3'] = aioner_out

#Wei2013
aioner_out = read_pubtator_format('results/AIONER/Wei2013_test_text_only.txt')[[0,3,4]]
aioner_out.columns = ['PMID','Mutation','Type']
aioner_out = aioner_out[aioner_out['Type']=='Variant']
AIONER['Wei2013'] = aioner_out

#SETH
aioner_out = read_pubtator_format('results/AIONER/SETH-corpus.pubtator')[[0,3,4]]
aioner_out.columns = ['PMID','Mutation','Type']
aioner_out = aioner_out[aioner_out['Type']=='Variant']
AIONER['SETH'] = aioner_out

#OSIRIS
aioner_out = read_pubtator_format('results/AIONER/OSIRIS_text_only.pubtator')[[0,3,4]]
aioner_out.columns = ['PMID','Mutation','Type']
aioner_out = aioner_out[aioner_out['Type']=='Variant']
AIONER['OSIRIS'] = aioner_out

AIONER.keys()

dict_keys(['tmVar3', 'Wei2013', 'SETH', 'OSIRIS'])

In [8]:
results = {}
details = {}

In [9]:
#PubMind
for key in GT.keys():
    #print(key)
    try:
        if 'dna' in key:
            detail,summary = dna_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(PM[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'PubMind-'+key)

        if 'pro' in key:
            detail,summary = protein_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(PM[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'PubMind-'+key)

        detail,summary = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(PM[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'PubMind-'+key)
        results['PubMind-'+key] = summary
        details['PubMind-'+key] = detail
    except Exception as e:
        print('Skip',key)

Skip tmVar3
## PubMind-tmVar3_dna benchmark (substring) | TP=524 FP=147 FN=234 | P=0.7809 R=0.6913 F1=0.7334 Acc(Jacc)=0.5790
## PubMind-tmVar3_dna benchmark (substring) | TP=524 FP=147 FN=234 | P=0.7809 R=0.6913 F1=0.7334 Acc(Jacc)=0.5790
## PubMind-tmVar3_pro benchmark (substring) | TP=326 FP=64 FN=179 | P=0.8359 R=0.6455 F1=0.7285 Acc(Jacc)=0.5729
## PubMind-tmVar3_pro benchmark (substring) | TP=326 FP=64 FN=179 | P=0.8359 R=0.6455 F1=0.7285 Acc(Jacc)=0.5729
Skip Wei2013
## PubMind-Wei2013_dna benchmark (substring) | TP=152 FP=39 FN=30 | P=0.7958 R=0.8352 F1=0.8150 Acc(Jacc)=0.6878
## PubMind-Wei2013_dna benchmark (substring) | TP=152 FP=39 FN=30 | P=0.7958 R=0.8352 F1=0.8150 Acc(Jacc)=0.6878
## PubMind-Wei2013_pro benchmark (substring) | TP=113 FP=16 FN=24 | P=0.8760 R=0.8248 F1=0.8496 Acc(Jacc)=0.7386
## PubMind-Wei2013_pro benchmark (substring) | TP=113 FP=16 FN=24 | P=0.8760 R=0.8248 F1=0.8496 Acc(Jacc)=0.7386
## PubMind-SETH benchmark (substring) | TP=618 FP=90 FN=80 | P=0.8729

In [10]:
#tmVar3
for key in GT.keys():
    #print(key)
    try:
        if 'dna' in key:
            detail,summary = dna_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(TV3[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'tmVar3-'+key)

        if 'pro' in key:
            detail,summary = protein_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(TV3[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'tmVar3-'+key)

        detail,summary = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(TV3[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'tmVar3-'+key)
        results['tmVar3-'+key] = summary
        details['tmVar3-'+key] = detail
    except:
        print('Skip',key)

Skip tmVar3
## tmVar3-tmVar3_dna benchmark (substring) | TP=596 FP=50 FN=162 | P=0.9226 R=0.7863 F1=0.8490 Acc(Jacc)=0.7376
## tmVar3-tmVar3_dna benchmark (substring) | TP=596 FP=50 FN=162 | P=0.9226 R=0.7863 F1=0.8490 Acc(Jacc)=0.7376
## tmVar3-tmVar3_pro benchmark (substring) | TP=474 FP=2 FN=31 | P=0.9958 R=0.9386 F1=0.9664 Acc(Jacc)=0.9349
## tmVar3-tmVar3_pro benchmark (substring) | TP=474 FP=2 FN=31 | P=0.9958 R=0.9386 F1=0.9664 Acc(Jacc)=0.9349
Skip Wei2013
## tmVar3-Wei2013_dna benchmark (substring) | TP=173 FP=35 FN=9 | P=0.8317 R=0.9505 F1=0.8872 Acc(Jacc)=0.7972
## tmVar3-Wei2013_dna benchmark (substring) | TP=173 FP=35 FN=9 | P=0.8317 R=0.9505 F1=0.8872 Acc(Jacc)=0.7972
## tmVar3-Wei2013_pro benchmark (substring) | TP=134 FP=14 FN=3 | P=0.9054 R=0.9781 F1=0.9404 Acc(Jacc)=0.8874
## tmVar3-Wei2013_pro benchmark (substring) | TP=134 FP=14 FN=3 | P=0.9054 R=0.9781 F1=0.9404 Acc(Jacc)=0.8874
## tmVar3-SETH benchmark (substring) | TP=638 FP=52 FN=60 | P=0.9246 R=0.9140 F1=0.9193

In [11]:
#SETH
for key in GT.keys():
    #print(key)
    try:
        if 'dna' in key:
            detail,summary = dna_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(SETH[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'SETH-'+key)

        if 'pro' in key:
            detail,summary = protein_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(SETH[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'SETH-'+key)

        detail,summary = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(SETH[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'SETH-'+key)
        results['SETH-'+key] = summary
        details['SETH-'+key] = detail
    except:
        print('Skip',key)

## SETH-tmVar3 benchmark (substring) | TP=793 FP=1 FN=470 | P=0.9987 R=0.6279 F1=0.7710 Acc(Jacc)=0.6274
Skip tmVar3_dna
Skip tmVar3_pro
## SETH-Wei2013 benchmark (substring) | TP=247 FP=4 FN=72 | P=0.9841 R=0.7743 F1=0.8667 Acc(Jacc)=0.7647
Skip Wei2013_dna
Skip Wei2013_pro
## SETH-SETH benchmark (substring) | TP=597 FP=7 FN=101 | P=0.9884 R=0.8553 F1=0.9171 Acc(Jacc)=0.8468
## SETH-OSIRIS benchmark (substring) | TP=213 FP=2 FN=157 | P=0.9907 R=0.5757 F1=0.7282 Acc(Jacc)=0.5726


In [12]:
#AIONER
for key in GT.keys():
    #print(key)
    try:
        if 'dna' in key:
            detail,summary = dna_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(AIONER[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'AIONER-'+key)

        if 'pro' in key:
            detail,summary = protein_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(AIONER[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'AIONER-'+key)

        detail,summary = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(AIONER[key]),
                                               clean_dnaNpro_mut_for_benchmark(GT[key]),
                                               'AIONER-'+key)
        results['AIONER-'+key] = summary
        details['AIONER-'+key] = detail
    except:
        print('Skip',key)

## AIONER-tmVar3 benchmark (substring) | TP=1246 FP=9 FN=17 | P=0.9928 R=0.9865 F1=0.9897 Acc(Jacc)=0.9796
Skip tmVar3_dna
Skip tmVar3_pro
## AIONER-Wei2013 benchmark (substring) | TP=316 FP=69 FN=3 | P=0.8208 R=0.9906 F1=0.8977 Acc(Jacc)=0.8144
Skip Wei2013_dna
Skip Wei2013_pro
## AIONER-SETH benchmark (substring) | TP=663 FP=51 FN=35 | P=0.9286 R=0.9499 F1=0.9391 Acc(Jacc)=0.8852
## AIONER-OSIRIS benchmark (substring) | TP=334 FP=14 FN=36 | P=0.9598 R=0.9027 F1=0.9304 Acc(Jacc)=0.8698


In [19]:
#adding manual adjusted result
def add_adj_result(key):
    print(key)
    reviewed_df = pd.read_csv(f"detail_benchmark_results_strict/{key}.csv")
    adj_detail, adj_summary = dnaorpro_benchmark_from_counts(reviewed_df, tool=key)
    results[key] = adj_summary
    details[key] = adj_detail

In [20]:
!ls detail_benchmark_results_strict/PubMind*adj.csv

detail_benchmark_results_strict/PubMind-tmVar3_dna-adj.csv
detail_benchmark_results_strict/PubMind-tmVar3_pro-adj.csv
detail_benchmark_results_strict/PubMind-Wei2013_dna-adj.csv
detail_benchmark_results_strict/PubMind-Wei2013_pro-adj.csv


In [21]:
add_adj_result('PubMind-tmVar3_dna-adj')

PubMind-tmVar3_dna-adj
## PubMind-tmVar3_dna-adj benchmark (from reviewed TP/FP/FN) | TP=595 FP=104 FN=162 | P=0.8512 R=0.7860 F1=0.8173 Acc(Jacc)=0.6911


In [22]:
add_adj_result('PubMind-tmVar3_pro-adj')

PubMind-tmVar3_pro-adj
## PubMind-tmVar3_pro-adj benchmark (from reviewed TP/FP/FN) | TP=434 FP=25 FN=72 | P=0.9455 R=0.8577 F1=0.8995 Acc(Jacc)=0.8173


In [23]:
add_adj_result('PubMind-Wei2013_dna-adj')

PubMind-Wei2013_dna-adj
## PubMind-Wei2013_dna-adj benchmark (from reviewed TP/FP/FN) | TP=163 FP=30 FN=19 | P=0.8446 R=0.8956 F1=0.8693 Acc(Jacc)=0.7689


In [24]:
add_adj_result('PubMind-Wei2013_pro-adj')

PubMind-Wei2013_pro-adj
## PubMind-Wei2013_pro-adj benchmark (from reviewed TP/FP/FN) | TP=126 FP=10 FN=11 | P=0.9265 R=0.9197 F1=0.9231 Acc(Jacc)=0.8571


In [26]:
#summarize the benchmark result
benchmark_results = results

rows = []

for key, val in benchmark_results.items():
    row = {
        "benchmark_name": val["tool"],
        "match_mode": val["match_mode"],
        "TP": val["micro"]["TP"],
        "FP": val["micro"]["FP"],
        "FN": val["micro"]["FN"],
        "micro_precision": val["micro"]["precision"],
        "micro_recall": val["micro"]["recall"],
        "micro_f1": val["micro"]["f1"],
        "micro_jaccard": val["micro"]["accuracy_jaccard"],
        "macro_precision": val["macro"]["precision"],
        "macro_recall": val["macro"]["recall"],
        "macro_f1": val["macro"]["f1"],
        "macro_jaccard": val["macro"]["accuracy_jaccard"],
    }
    rows.append(row)

#build DataFrame
df = pd.DataFrame(rows)


In [27]:
# Display
col_order=list(df.columns.drop('benchmark_name','match_mode'))
# Split only on the first dash
df[["tool", "corpus"]] = df["benchmark_name"].str.split("-", n=1, expand=True)

df = df[["tool", "corpus"]+col_order]
df=df.sort_values('tool')
df

,tool,corpus,match_mode,TP,FP,FN,micro_precision,micro_recall,micro_f1,micro_jaccard,macro_precision,macro_recall,macro_f1,macro_jaccard
19,AIONER,OSIRIS,substring,334,14,36,0.959770,0.902703,0.930362,0.869792,0.922730,0.893338,0.896880,0.868017
18,AIONER,SETH,substring,663,51,35,0.928571,0.949857,0.939093,0.885180,0.901033,0.945009,0.908162,0.881755
17,AIONER,Wei2013,substring,316,69,3,0.820779,0.990596,0.897727,0.814433,0.794967,0.985699,0.858927,0.792217
16,AIONER,tmVar3,substring,1246,9,17,0.992829,0.986540,0.989674,0.979560,0.990735,0.984039,0.984725,0.977482
0,PubMind,tmVar3_dna,substring,524,147,234,0.780924,0.691293,0.733380,0.579006,0.751569,0.684172,0.688729,0.629346
21,PubMind,tmVar3_pro-adj,substring+manual_adjusted,434,25,72,0.945534,0.857708,0.899482,0.817326,0.860188,0.829200,0.835888,0.812073
20,PubMind,tmVar3_dna-adj,substring+manual_adjusted,595,104,162,0.851216,0.785997,0.817308,0.691057,0.846699,0.811725,0.804165,0.754369
22,PubMind,Wei2013_dna-adj,substring+manual_adjusted,163,30,19,0.844560,0.895604,0.869333,0.768868,0.848951,0.898467,0.846537,0.804890
23,PubMind,Wei2013_pro-adj,substring+manual_adjusted,126,10,11,0.926471,0.919708,0.923077,0.857143,0.908506,0.886218,0.889513,0.871647
5,PubMind,OSIRIS,substring,263,38,107,0.873754,0.710811,0.783905,0.644608,0.862426,0.724082,0.749708,0.678876


## Strict result
the strict result keep the "-" in the mutation (mainly for easier manual check for PubMind
Also the strict result will remove the entire comparison for PMID that if the ground truth is empty set. For example, if GT did not have any mutation, this row will be removed completely even though tool found mutations.

In [28]:
# ---- Save results ----
df.to_csv("benchmark_final_table_strict.tsv", sep="\t", index=False)
df.to_csv("benchmark_final_table_strict.csv", index=False)

print("Saved as:")
print(" - benchmark_final_table_strict.tsv")
print(" - benchmark_final_table_strict.csv")

Saved as:
 - benchmark_final_table_strict.tsv
 - benchmark_final_table_strict.csv


In [29]:
df.sort_values('macro_f1',ascending=False)

,tool,corpus,match_mode,TP,FP,FN,micro_precision,micro_recall,micro_f1,micro_jaccard,macro_precision,macro_recall,macro_f1,macro_jaccard
16,AIONER,tmVar3,substring,1246,9,17,0.992829,0.986540,0.989674,0.979560,0.990735,0.984039,0.984725,0.977482
7,tmVar3,tmVar3_pro,substring,474,2,31,0.995798,0.938614,0.966361,0.934911,0.932870,0.917361,0.922276,0.915046
18,AIONER,SETH,substring,663,51,35,0.928571,0.949857,0.939093,0.885180,0.901033,0.945009,0.908162,0.881755
19,AIONER,OSIRIS,substring,334,14,36,0.959770,0.902703,0.930362,0.869792,0.922730,0.893338,0.896880,0.868017
23,PubMind,Wei2013_pro-adj,substring+manual_adjusted,126,10,11,0.926471,0.919708,0.923077,0.857143,0.908506,0.886218,0.889513,0.871647
9,tmVar3,Wei2013_pro,substring,134,14,3,0.905405,0.978102,0.940351,0.887417,0.846612,0.955128,0.881213,0.840201
17,AIONER,Wei2013,substring,316,69,3,0.820779,0.990596,0.897727,0.814433,0.794967,0.985699,0.858927,0.792217
22,PubMind,Wei2013_dna-adj,substring+manual_adjusted,163,30,19,0.844560,0.895604,0.869333,0.768868,0.848951,0.898467,0.846537,0.804890
8,tmVar3,Wei2013_dna,substring,173,35,9,0.831731,0.950549,0.887179,0.797235,0.806992,0.924330,0.845947,0.802969
10,tmVar3,SETH,substring,638,52,60,0.924638,0.914040,0.919308,0.850667,0.841126,0.881022,0.845557,0.808700


In [16]:
os.makedirs('detail_benchmark_results_strict',exist_ok=True)

In [17]:

# Save details
for key in details.keys():
    print(key)
    details[key].to_csv(f'detail_benchmark_results_strict/{key}.tsv',
                        sep='\t',index=False,header=True)


PubMind-tmVar3_dna
PubMind-tmVar3_pro
PubMind-Wei2013_dna
PubMind-Wei2013_pro
PubMind-SETH
PubMind-OSIRIS
tmVar3-tmVar3_dna
tmVar3-tmVar3_pro
tmVar3-Wei2013_dna
tmVar3-Wei2013_pro
tmVar3-SETH
tmVar3-OSIRIS
SETH-tmVar3
SETH-Wei2013
SETH-SETH
SETH-OSIRIS
AIONER-tmVar3
AIONER-Wei2013
AIONER-SETH
AIONER-OSIRIS


In [24]:
# ---- Save results ----
df.to_csv("benchmark_final_table.tsv", sep="\t", index=False)
df.to_csv("benchmark_final_table.csv", index=False)

print("Saved as:")
print(" - benchmark_final_table.tsv")
print(" - benchmark_final_table.csv")

Saved as:
 - benchmark_final_table.tsv
 - benchmark_final_table.csv


In [26]:
df.sort_values('macro_f1',ascending=False)

,tool,corpus,match_mode,TP,FP,FN,micro_precision,micro_recall,micro_f1,micro_jaccard,macro_precision,macro_recall,macro_f1,macro_jaccard
16,AIONER,tmVar3,substring,1246,19,17,0.984980,0.986540,0.985759,0.971919,0.990735,0.984039,0.984725,0.977482
7,tmVar3,tmVar3_pro,substring,474,8,31,0.983402,0.938614,0.960486,0.923977,0.932870,0.917361,0.922276,0.915046
18,AIONER,SETH,substring,663,99,35,0.870079,0.949857,0.908219,0.831870,0.901033,0.945009,0.908162,0.881755
19,AIONER,OSIRIS,substring,334,16,36,0.954286,0.902703,0.927778,0.865285,0.922730,0.893338,0.896880,0.868017
9,tmVar3,Wei2013_pro,substring,134,18,3,0.881579,0.978102,0.927336,0.864516,0.846612,0.955128,0.881213,0.840201
17,AIONER,Wei2013,substring,316,96,3,0.766990,0.990596,0.864569,0.761446,0.794967,0.985699,0.858927,0.792217
8,tmVar3,Wei2013_dna,substring,173,91,9,0.655303,0.950549,0.775785,0.633700,0.806992,0.924330,0.845947,0.802969
10,tmVar3,SETH,substring,638,346,60,0.648374,0.914040,0.758621,0.611111,0.841126,0.881022,0.845557,0.808700
4,PubMind,SETH,substring,618,194,80,0.761084,0.885387,0.818543,0.692825,0.840898,0.848408,0.830060,0.796355
14,SETH,SETH,substring,597,34,101,0.946117,0.855301,0.898420,0.815574,0.864773,0.818410,0.829939,0.808183
